# Gold: fact_order_items

## Import Helper Functions

In [1]:
from src.config_loader import load_config
from src.spark_sql_magic import sql
from src.gold.helper import get_changed_customer_ids, get_changed_order_ids, get_changed_product_ids, get_changed_seller_ids
from src.gold.facts.order_items import build_fact_order_items, build_fact_order_items_incremental, validate_fact_order_items
from src.monitoring import write_dq_metrics
from src.watermark import get_last_commit_ts, get_effective_watermark
from src.writers import overwrite_table, replace_by_key

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Load Configs

In [2]:
cfg = load_config()

JOB_NAMES = cfg["spark_jobs"]["jobs"]
CATALOG = cfg["general"]["catalog"]
SILVER_NAMESPACE = cfg["general"]["namespaces"]["silver"]
GOLD_NAMESPACE = cfg["general"]["namespaces"]["gold"]

cfg_order_items= cfg["gold"]["fact_order_items"]
SOURCE_TABLE = cfg_order_items["source_table"]
TARGET_TABLE = cfg_order_items["target_table"]
ORDERS_TABLE = cfg_order_items["orders_table"]
CUSTOMERS_TABLE = cfg_order_items["customers_table"]
PRODUCTS_TABLE = cfg_order_items["products_table"]
SELLERS_TABLE = cfg_order_items["sellers_table"]
DATE_TABLE = cfg_order_items["date_table"]
BUFFER_HOURS = cfg_order_items["buffer_hours"]
KEY_COLUMNS = cfg_order_items["key_columns"]

## Import Libraries and Start Session

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["gold_order_items"])
        .getOrCreate()
)

## Run Pipeline

In [4]:
def run_fact_order_items_pipeline(spark):
    print("[START] fact_order_items pipeline")
    last_commit_ts = get_last_commit_ts(spark, TARGET_TABLE)
    print(f"[INFO] last_commit_ts = {last_commit_ts}")

    if last_commit_ts is None:
        print("[INFO] first run → full rebuild")
        df = build_fact_order_items(
            spark,
            SOURCE_TABLE,
            ORDERS_TABLE, 
            CUSTOMERS_TABLE,
            SELLERS_TABLE,
            PRODUCTS_TABLE, 
            DATE_TABLE
        )
        metrics = validate_fact_order_items(df)
        write_dq_metrics(df, metrics, "fact_order_items", GOLD_NAMESPACE, "monitoring.dq_metrics")
        overwrite_table(df, TARGET_TABLE)

    effective_ts = get_effective_watermark(last_commit_ts, BUFFER_HOURS)
    changed_order_ids = get_changed_order_ids(spark, effective_ts)
    changed_customer_ids = get_changed_customer_ids(spark, effective_ts)
    changed_seller_ids = get_changed_seller_ids(spark, effective_ts)
    changed_product_ids = get_changed_product_ids(spark, effective_ts)
    
    if changed_customer_ids.isEmpty() and changed_order_ids.isEmpty() and changed_product_ids.isEmpty() and changed_seller_ids.isEmpty():
        print("[INFO] no changes detected → skip")
        return

    print("[INFO] changes detected → incremental run")
    df = build_fact_order_items_incremental(
        spark,
        SOURCE_TABLE,
        ORDERS_TABLE, 
        CUSTOMERS_TABLE,
        SELLERS_TABLE,
        PRODUCTS_TABLE, 
        DATE_TABLE,
        changed_order_ids,
        changed_customer_ids,
        changed_seller_ids,
        changed_product_ids
    )
    metrics = validate_fact_order_items(df)
    write_dq_metrics(df, metrics, "fact_order_items", GOLD_NAMESPACE, "monitoring.dq_metrics")
    replace_by_key(spark, df, TARGET_TABLE, KEY_COLUMNS)
    print("[END] incremental update complete")

In [5]:
# if __name__ == "__main__":
#     from pyspark.sql import SparkSession

#     spark = SparkSession.builder.getOrCreate()
run_fact_order_items_pipeline(spark)

[START] fact_order_items pipeline
[INFO] last_commit_ts = 2026-04-06 08:32:02.002000


[INFO] changes detected → incremental run


[END] incremental update complete


## Sanity Check

In [6]:
%%sql
SHOW TABLES IN polaris.gold;

+---------+------------------+-----------+
|namespace|tableName         |isTemporary|
+---------+------------------+-----------+
|gold     |fact_orders       |false      |
|gold     |dim_date          |false      |
|gold     |dim_customers_scd2|false      |
|gold     |dim_sellers_scd2  |false      |
|gold     |dim_products_scd2 |false      |
|gold     |fact_order_items  |false      |
+---------+------------------+-----------+



In [7]:
%%sql
SELECT * FROM polaris.gold.fact_order_items
LIMIT 10

+--------------------------------+-------------+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+--------------------------------+--------------------------------+--------------------------------+-------------------+----------+-------------+------------+----------------------+------------------------+-------------------+----------------------------+--------------------------------+-----------------------------+-----------------------------+
|order_id                        |order_item_id|customer_sk                                                     |seller_sk                                                       |product_sk                                                      |customer_id                     |seller_id                       |product_id                      |shipping_limit_date|price     |freight_value|order_status|order_pur

In [8]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop() 